# Customer Revenue & Retention Analytics

A reproducible companion to the layered DuckDB model. The notebook answers a commercial prioritisation question; it is not the pipeline itself.

## TL;DR

The validated sample contains 59 customers, 412 invoices and $2,328.60 revenue. As of 2025-12-31, 12 customers are in the 91–180 day At-risk band. The United States and Rock are the largest market and genre contributors.

## Context and methods

Decision: identify revenue drivers and a defensible re-engagement audience. SQL creates governed raw, dimensional, fact, lifecycle and KPI layers. Inactivity risk is a proxy, not confirmed churn.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from jira_analytics.pipeline import build
con = build(root=ROOT)
sns.set_theme(style='whitegrid')
print('Pipeline ready')

Pipeline ready


## Data

Chinook is an MIT-licensed relational sample database. Customers and sales are fictional/generated. Direct identifiers are excluded from this committed snapshot; the analytical period is 2021-01-01 to 2025-12-22.

In [2]:
quality = con.execute('SELECT * FROM audit.data_quality_results').df()
quality

                           check_name observed_value status
0                 customers_row_count             59   PASS
1                  invoices_row_count            412   PASS
2             invoice_lines_row_count           2240   PASS
3             duplicate_customer_keys              0   PASS
4              duplicate_invoice_keys              0   PASS
5         duplicate_invoice_line_keys              0   PASS
6          invoice_customer_orphans              0   PASS
7              line_invoice_orphans              0   PASS
8                line_track_orphans              0   PASS
9              null_required_fields              0   PASS
10          nonpositive_line_values              0   PASS
11        invoice_line_reconciliation              0   PASS

### Quality gate

All required row-count, uniqueness, referential-integrity, required-field and financial-reconciliation checks must pass. The pipeline raises an error if any check fails.

In [3]:
headline = con.execute('''
SELECT ROUND(SUM(total), 2) AS revenue, COUNT(*) AS orders,
       COUNT(DISTINCT customer_id) AS customers,
       ROUND(SUM(total) / COUNT(*), 2) AS average_order_value
FROM analytics.fact_invoice
''').df()
headline

   revenue  orders  customers  average_order_value
0   2328.6     412         59                 5.65

## Results

The following views separate customer recency from revenue mix, so a re-engagement decision is not based on aggregate revenue alone.

In [4]:
status = con.execute('''
SELECT activity_status, COUNT(*) AS customers
FROM analytics.customer_lifecycle GROUP BY activity_status
ORDER BY CASE activity_status WHEN 'Active' THEN 1 WHEN 'At risk' THEN 2 ELSE 3 END
''').df()
status

  activity_status  customers
0          Active         19
1         At risk         12
2          Dormant         28

### Revenue trend

Monthly revenue is shown with a three-month rolling total. Missing calendar months remain in the date spine rather than disappearing silently.

In [5]:
monthly = con.execute('SELECT * FROM analytics.monthly_kpis ORDER BY month_start').df()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pd.to_datetime(monthly['month_start']), monthly['revenue'], color='#2563EB', linewidth=2, label='Monthly revenue')
ax.plot(pd.to_datetime(monthly['month_start']), monthly['rolling_3m_revenue'], color='#14B8A6', linewidth=2, label='Rolling 3-month revenue')
ax.set(title='Monthly revenue and rolling three-month total', xlabel='', ylabel='Revenue ($)')
ax.legend(frameon=False)
sns.despine()
fig.tight_layout()
fig

<Figure size 1000x400 with 1 Axes>

### Market and product mix

Country and genre rankings are derived from joined invoice-line, invoice and product dimensions—not from unrelated single-table averages.

In [6]:
leaders = con.execute('''
SELECT c.country AS top_country, c.revenue AS country_revenue,
       p.genre_name AS top_genre, p.revenue AS genre_revenue
FROM (SELECT * FROM analytics.country_performance ORDER BY revenue DESC LIMIT 1) c
CROSS JOIN (SELECT * FROM analytics.product_performance ORDER BY revenue DESC LIMIT 1) p
''').df()
leaders

  top_country  country_revenue top_genre  genre_revenue
0         USA           523.06      Rock         826.65

## Takeaways

1. Start a targeted re-engagement experiment with the 12 At-risk customers; do not mix them with the 28 Dormant customers.
2. Protect the strongest revenue concentrations—US customers and Rock—while testing whether the pattern holds in real operational data.
3. Treat 100% repeat-buyer rates in 2025 as a property of generated sample data, not a benchmark.
4. Before production, add real cancellation/subscription state, campaign exposure, margin and calibrated purchase-cadence thresholds.